# S08 — Testing, Type Hints, Exceptions & Defensive Programming

**Python for Applied Physics** | Master in Applied Physics  
**Prerequisites:** S01–S07

---

## Learning objectives

By the end of this session you will be able to:

- Annotate functions and classes with type hints and check them with `mypy`
- Raise, catch, and chain exceptions; define custom exception classes
- Write `pytest` unit tests with `approx`, `raises`, fixtures, and `parametrize`
- Test class hierarchies and validate exception behaviour
- Apply defensive programming patterns: guard clauses, `warnings`, logging
- Measure test coverage with `pytest-cov`

---

## Session map

| Block | Topic | Cells |
|-------|-------|-------|
| 1 | Why test? | 1–3 |
| 2 | Type hints | 4–8 |
| 3 | Exceptions | 9–13 |
| 4 | `pytest` basics | 14–19 |
| 5 | Fixtures and parametrize | 20–24 |
| 6 | Testing classes | 25–28 |
| 7 | Defensive programming | 29–33 |
| 8 | Coverage and CI preview | 34–36 |

In [1]:
import numpy as np
import warnings
import logging
from typing import Optional, Union

print(f"NumPy {np.__version__}")

NumPy 2.4.3


---
## Block 1 — Why test?

Numerical code fails silently. A wrong sign, a missing factor of 2, a transposed index — none of these raise exceptions. They produce plausible-looking numbers that are quietly wrong.

**The cost of not testing in physics:**
- Published results that cannot be reproduced
- Experiments designed around incorrect simulations
- Weeks of debugging after a refactor

**The test pyramid:**
```
         ▲  Integration tests  (few, slow)
        ▲▲▲  Unit tests        (many, fast)
```

Unit tests check one function or method in isolation.  
Integration tests check that components work together.

In [3]:
# --------------------------------------------------------------------------
# 🔬 Physics: a silent bug in the Fresnel equations
# --------------------------------------------------------------------------

def r_s_buggy(n1: float, n2: float, theta_i: float) -> float:
    """s-polarisation reflection coefficient — contains a sign bug."""
    cos_i = np.cos(theta_i)
    sin_t = n1 / n2 * np.sin(theta_i)
    cos_t = np.sqrt(1 - sin_t**2)
    return (n1 * cos_i + n2 * cos_t) / (n1 * cos_i - n2 * cos_t)  # ← sign flipped

def r_s_correct(n1: float, n2: float, theta_i: float) -> float:
    """s-polarisation reflection coefficient — correct."""
    cos_i = np.cos(theta_i)
    sin_t = n1 / n2 * np.sin(theta_i)
    cos_t = np.sqrt(1 - sin_t**2)
    return (n1 * cos_i - n2 * cos_t) / (n1 * cos_i + n2 * cos_t)

n1, n2 = 1.0, 1.5
theta  = np.radians(45)

print(f"r_s (buggy)  : {r_s_buggy(n1, n2, theta):.6f}")
print(f"r_s (correct): {r_s_correct(n1, n2, theta):.6f}")
print()
print("Both return a number. Only a test reveals the bug:")
print(f"  At normal incidence (θ=0): r_s = (n1-n2)/(n1+n2) = {(n1-n2)/(n1+n2):.6f}")
print(f"  Buggy at θ=0: {r_s_buggy(n1, n2, 0):.6f}")
print(f"  Correct at θ=0: {r_s_correct(n1, n2, 0):.6f}")

r_s (buggy)  : -3.296663
r_s (correct): -0.303337

Both return a number. Only a test reveals the bug:
  At normal incidence (θ=0): r_s = (n1-n2)/(n1+n2) = -0.200000
  Buggy at θ=0: -5.000000
  Correct at θ=0: -0.200000


In [4]:
# The test that would have caught it:

def test_r_s_normal_incidence():
    """At θ=0, r_s = (n1-n2)/(n1+n2) — Fresnel analytic result."""
    n1, n2 = 1.0, 1.5
    expected = (n1 - n2) / (n1 + n2)
    result   = r_s_correct(n1, n2, theta_i=0.0)
    assert abs(result - expected) < 1e-10, f"{result} ≠ {expected}"

def test_r_s_buggy_fails():
    n1, n2   = 1.0, 1.5
    expected = (n1 - n2) / (n1 + n2)
    result   = r_s_buggy(n1, n2, theta_i=0.0)
    # This assertion FAILS — exposing the bug
    return abs(result - expected) < 1e-10

test_r_s_normal_incidence()
print(f"test_r_s_normal_incidence passed ✓")
print(f"test_r_s_buggy would fail: {test_r_s_buggy_fails()}")

test_r_s_normal_incidence passed ✓
test_r_s_buggy would fail: False


---
## Block 2 — Type hints

Type hints annotate what types a function expects and returns. They are:
- **Not enforced at runtime** — Python ignores them during execution
- **Documentation** — instantly communicates intent
- **Tooling input** — `mypy`, Pylance, and VS Code use them for static analysis and autocomplete

In [3]:
# --------------------------------------------------------------------------
# Basic annotations
# --------------------------------------------------------------------------

from __future__ import annotations   # allows forward references without quotes
from typing import Optional, Union

def gaussian_intensity(
    r: np.ndarray,
    w: float,
    P: float,
) -> np.ndarray:
    """
    Radial intensity profile of a Gaussian beam.

    Parameters
    ----------
    r : np.ndarray  — radial coordinate (m)
    w : float       — beam radius 1/e² (m)
    P : float       — total power (W)

    Returns
    -------
    np.ndarray — intensity in W/m²
    """
    I0 = 2 * P / (np.pi * w**2)
    return I0 * np.exp(-2 * r**2 / w**2)


def fit_gaussian(
    r: np.ndarray,
    I: np.ndarray,
    p0: Optional[list[float]] = None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Fit a Gaussian to a beam profile.

    Returns
    -------
    (popt, pcov) — optimal parameters and covariance matrix
    """
    from scipy.optimize import curve_fit

    def model(r, I0, w, r0):
        return I0 * np.exp(-2 * (r - r0)**2 / w**2)

    if p0 is None:
        p0 = [I.max(), (r[-1] - r[0]) / 4, 0.0]

    return curve_fit(model, r, I, p0=p0)


r_test = np.linspace(-1e-3, 1e-3, 100)
I_test = gaussian_intensity(r_test, w=300e-6, P=1.0)
print(f"gaussian_intensity: {I_test.shape}, max={I_test.max():.2f} W/m²")

gaussian_intensity: (100,), max=7057533.03 W/m²


In [4]:
# --------------------------------------------------------------------------
# Optional, Union, list[T]
# --------------------------------------------------------------------------

def pulse_energy(
    t: np.ndarray,
    E_t: np.ndarray,
    dt: Optional[float] = None,
) -> float:
    """Pulse energy U = ∫|E(t)|² dt."""
    if dt is None:
        dt = t[1] - t[0]
    return float(np.trapz(np.abs(E_t)**2, t))


def make_grid(
    limits: Union[float, tuple[float, float]],
    N: int,
) -> np.ndarray:
    """Build a 1D grid. limits can be a single half-width or (min, max)."""
    if isinstance(limits, (int, float)):
        return np.linspace(-limits, limits, N)
    return np.linspace(limits[0], limits[1], N)


print(make_grid(1e-3, 5))
print(make_grid((-2e-3, 3e-3), 5))

[-0.001  -0.0005  0.      0.0005  0.001 ]
[-0.002   -0.00075  0.0005   0.00175  0.003  ]


In [5]:
# --------------------------------------------------------------------------
# Type hints in classes — forward references with from __future__ import annotations
# --------------------------------------------------------------------------

class GaussianPulse:
    c: float = 3e8

    def __init__(self, tau: float, wavelength: float, energy: float) -> None:
        self._tau        = tau
        self._wavelength = wavelength
        self._energy     = energy

    @property
    def tau(self) -> float: return self._tau

    @property
    def wavelength(self) -> float: return self._wavelength

    @property
    def energy(self) -> float: return self._energy

    def intensity(self, t: np.ndarray) -> np.ndarray:
        return np.exp(-t**2 / self._tau**2)

    def copy(self) -> GaussianPulse:   # forward reference — works with __future__ import
        return GaussianPulse(self._tau, self._wavelength, self._energy)

    def __repr__(self) -> str:
        return f"GaussianPulse(tau={self._tau*1e15:.1f} fs)"


p = GaussianPulse(100e-15, 800e-9, 50e-6)
p2 = p.copy()
print(p, p2)

GaussianPulse(tau=100.0 fs) GaussianPulse(tau=100.0 fs)


⚠️ **Common Pitfall — type hints are not enforced at runtime**

```python
def gaussian_intensity(r: np.ndarray, w: float, P: float) -> np.ndarray:
    ...

gaussian_intensity(0.5, 'wrong', None)   # No error at runtime!
```

Type hints only affect:
- Static analysis tools (`mypy`, Pylance)
- IDE autocomplete and warnings

For runtime enforcement use `@property` setters with `isinstance` checks, or `isinstance` guard clauses at the top of functions.

**Running mypy from the terminal:**
```bash
pip install mypy
mypy shared/optics.py --ignore-missing-imports
```

In [ ]:
# --------------------------------------------------------------------------
# Protocol — structural typing (duck typing formalised)
# --------------------------------------------------------------------------

from typing import Protocol

class HasIntensity(Protocol):
    """Any object that has an intensity(t) method."""
    def intensity(self, t: np.ndarray) -> np.ndarray: ...


def compute_energy(pulse: HasIntensity, t: np.ndarray) -> float:
    """
    Works with any object implementing intensity(t),
    not just GaussianPulse — structural (duck) typing.
    """
    I_t = pulse.intensity(t)
    return float(np.trapz(I_t, t))


t  = np.linspace(-1e-12, 1e-12, 1024)
en = compute_energy(p, t)
print(f"Energy (from protocol): {en:.4e}")

---
## Block 3 — Exceptions

Exceptions are Python's mechanism for signalling and handling errors. They are objects in a hierarchy:

```
BaseException
├── KeyboardInterrupt
├── SystemExit
└── Exception
    ├── ValueError      — wrong value for the type
    ├── TypeError       — wrong type entirely
    ├── RuntimeError    — something went wrong at runtime
    ├── FileNotFoundError
    ├── ZeroDivisionError
    └── ... (and your custom exceptions)
```

In [6]:
# --------------------------------------------------------------------------
# raise, try/except/else/finally
# --------------------------------------------------------------------------

def safe_sqrt(x: float) -> float:
    """Return √x, raising ValueError for negative input."""
    if x < 0:
        raise ValueError(f"Cannot take sqrt of negative number: {x}")
    return float(np.sqrt(x))


# try / except / else / finally
for value in [4.0, -1.0, 0.0]:
    try:
        result = safe_sqrt(value)
    except ValueError as e:
        print(f"ValueError for {value}: {e}")
    else:
        # Runs only if no exception was raised
        print(f"sqrt({value}) = {result}")
    finally:
        # Always runs — use for cleanup (close files, release resources)
        pass

sqrt(4.0) = 2.0
ValueError for -1.0: Cannot take sqrt of negative number: -1.0
sqrt(0.0) = 0.0


In [ ]:
# --------------------------------------------------------------------------
# Catching multiple exception types
# --------------------------------------------------------------------------

def load_beam_data(path: str) -> np.ndarray:
    """Load a 1D beam profile from a text file."""
    try:
        data = np.loadtxt(path)
    except FileNotFoundError:
        raise FileNotFoundError(f"Beam data file not found: {path!r}")
    except ValueError as e:
        raise ValueError(f"Could not parse {path!r}: {e}") from e
    if data.ndim != 1:
        raise ValueError(f"Expected 1D array, got shape {data.shape}")
    return data


# raise ... from ... — exception chaining preserves context
try:
    load_beam_data('/nonexistent/beam.txt')
except FileNotFoundError as e:
    print(f"Caught: {e}")

In [ ]:
# --------------------------------------------------------------------------
# Custom exception classes
# --------------------------------------------------------------------------

class PhysicsError(Exception):
    """Base class for physics-domain errors in this course."""


class PulseError(PhysicsError, ValueError):
    """
    Raised when pulse parameters are unphysical.

    Inherits from both PhysicsError (for domain-specific catching)
    and ValueError (so generic code that catches ValueError still works).
    """
    def __init__(self, parameter: str, value: float, reason: str) -> None:
        self.parameter = parameter
        self.value     = value
        super().__init__(f"{parameter}={value}: {reason}")


class ConvergenceError(PhysicsError, RuntimeError):
    """Raised when an iterative algorithm fails to converge."""
    def __init__(self, algorithm: str, iterations: int, tolerance: float) -> None:
        super().__init__(
            f"{algorithm} did not converge after {iterations} iterations "
            f"(tolerance={tolerance:.2e})"
        )


# Use in GaussianPulse
class GaussianPulse:
    def __init__(self, tau: float, wavelength: float, energy: float) -> None:
        if tau <= 0:
            raise PulseError('tau', tau, 'must be positive')
        if wavelength <= 0:
            raise PulseError('wavelength', wavelength, 'must be positive')
        if energy < 0:
            raise PulseError('energy', energy, 'must be non-negative')
        self._tau = tau; self._wavelength = wavelength; self._energy = energy

    @property
    def tau(self) -> float: return self._tau
    @property
    def wavelength(self) -> float: return self._wavelength
    @property
    def energy(self) -> float: return self._energy

    def intensity(self, t: np.ndarray) -> np.ndarray:
        return np.exp(-t**2 / self._tau**2)

    def __repr__(self) -> str:
        return f"GaussianPulse(tau={self._tau*1e15:.1f} fs, λ={self._wavelength*1e9:.0f} nm)"


# Custom exception carries structured information
try:
    GaussianPulse(tau=-1e-15, wavelength=800e-9, energy=50e-6)
except PulseError as e:
    print(f"PulseError: {e}")
    print(f"  parameter = {e.parameter}")
    print(f"  value     = {e.value}")

# PulseError IS-A ValueError — caught by generic code
try:
    GaussianPulse(tau=-1e-15, wavelength=800e-9, energy=50e-6)
except ValueError as e:
    print(f"Also caught as ValueError: {e}")

⚠️ **Common Pitfall — bare `except:`**

```python
# WRONG: catches KeyboardInterrupt, SystemExit — impossible to stop the script
try:
    result = compute_something()
except:
    print("Something went wrong")

# RIGHT: catch specific types
try:
    result = compute_something()
except (ValueError, RuntimeError) as e:
    print(f"Computation failed: {e}")
```

If you truly need to catch everything (e.g. in a server loop), use `except Exception` — this still lets `KeyboardInterrupt` and `SystemExit` through.

In [ ]:
# ⚡ Try It — exception chaining
#
# Write a function load_pulse_params(path: str) -> dict that:
# 1. Reads a JSON file
# 2. Raises FileNotFoundError if the file does not exist
# 3. Raises PulseError (chained from json.JSONDecodeError) if the file is malformed
# 4. Raises PulseError if 'tau_fs' key is missing or non-positive

import json

def load_pulse_params(path: str) -> dict:
    # YOUR CODE HERE
    pass

# Solution (hidden)
# def load_pulse_params(path: str) -> dict:
#     try:
#         with open(path) as f:
#             data = json.load(f)
#     except FileNotFoundError:
#         raise
#     except json.JSONDecodeError as e:
#         raise PulseError('file', 0, f'malformed JSON in {path!r}') from e
#     if 'tau_fs' not in data:
#         raise PulseError('tau_fs', 0, 'key missing from config')
#     if data['tau_fs'] <= 0:
#         raise PulseError('tau_fs', data['tau_fs'], 'must be positive')
#     return data

---
## Block 4 — `pytest` basics

`pytest` discovers and runs tests automatically. You write plain functions whose names start with `test_` and use `assert`.

```bash
# Install
pip install pytest pytest-cov

# Run all tests
pytest

# Verbose output
pytest -v

# Run only tests matching a pattern
pytest -k "fresnel"

# Stop after first failure
pytest -x
```

**Naming conventions:**
- Test files: `test_optics.py`, `test_pulses.py`
- Test functions: `def test_gaussian_intensity_peak():`
- One assertion per test — easier to diagnose failures

In [ ]:
# --------------------------------------------------------------------------
# Writing tests — in a notebook cell first, then copy to test_optics.py
# --------------------------------------------------------------------------

import pytest

# The function under test
def gaussian_intensity(r: np.ndarray, w: float, P: float) -> np.ndarray:
    I0 = 2 * P / (np.pi * w**2)
    return I0 * np.exp(-2 * r**2 / w**2)


# ---------- tests ----------

def test_gaussian_intensity_peak():
    """Peak at r=0 equals 2P/(πw²)."""
    w, P = 500e-6, 1.0
    I0   = gaussian_intensity(np.array([0.0]), w=w, P=P)[0]
    assert I0 == pytest.approx(2 * P / (np.pi * w**2), rel=1e-6)


def test_gaussian_intensity_at_w():
    """At r=w the intensity is I₀·exp(-2)."""
    w, P = 500e-6, 1.0
    I0   = 2 * P / (np.pi * w**2)
    I_w  = gaussian_intensity(np.array([w]), w=w, P=P)[0]
    assert I_w == pytest.approx(I0 * np.exp(-2), rel=1e-6)


def test_gaussian_intensity_integrated_power():
    """2π ∫ I(r) r dr = P."""
    w, P  = 500e-6, 1.0
    r     = np.linspace(0, 5*w, 10000)
    I     = gaussian_intensity(r, w=w, P=P)
    P_int = 2 * np.pi * np.trapz(I * r, r)
    assert P_int == pytest.approx(P, rel=1e-3)


def test_gaussian_intensity_symmetry():
    """I(r) == I(-r) — radial symmetry."""
    w, P = 500e-6, 1.0
    r    = np.linspace(-3*w, 3*w, 256)
    I    = gaussian_intensity(r, w=w, P=P)
    assert np.allclose(I, I[::-1])


# Run the tests here in the notebook
test_gaussian_intensity_peak()
test_gaussian_intensity_at_w()
test_gaussian_intensity_integrated_power()
test_gaussian_intensity_symmetry()
print("All tests passed ✓")

In [ ]:
# --------------------------------------------------------------------------
# pytest.raises — testing exceptions
# --------------------------------------------------------------------------

def test_pulse_error_negative_tau():
    """GaussianPulse raises PulseError for negative tau."""
    with pytest.raises(PulseError, match='must be positive'):
        GaussianPulse(tau=-1e-15, wavelength=800e-9, energy=50e-6)


def test_pulse_error_is_value_error():
    """PulseError is also a ValueError — caught by generic code."""
    with pytest.raises(ValueError):
        GaussianPulse(tau=-1e-15, wavelength=800e-9, energy=50e-6)


def test_pulse_valid_zero_energy():
    """Energy=0 is valid (e.g. before amplification)."""
    p = GaussianPulse(100e-15, 800e-9, energy=0.0)
    assert p.energy == 0.0


test_pulse_error_negative_tau()
test_pulse_error_is_value_error()
test_pulse_valid_zero_energy()
print("Exception tests passed ✓")

In [ ]:
# --------------------------------------------------------------------------
# pytest.approx — floating-point comparison
# --------------------------------------------------------------------------

# Never do this in tests:
# assert 0.1 + 0.2 == 0.3   # False! floating-point

# Use pytest.approx:
assert 0.1 + 0.2 == pytest.approx(0.3)             # relative tolerance 1e-6
assert 1e-15 == pytest.approx(1e-15, rel=1e-4)      # custom relative
assert 1e-15 == pytest.approx(1.001e-15, rel=1e-2)  # 1% tolerance

# Works with arrays too
a = np.array([1.0, 2.0, 3.0])
b = a + 1e-10
assert a == pytest.approx(b, abs=1e-8)

print("pytest.approx examples passed ✓")

---
## Block 5 — Fixtures and parametrize

In [ ]:
# --------------------------------------------------------------------------
# @pytest.fixture — shared setup, injected by name
# --------------------------------------------------------------------------

# In a real test file this would be decorated with @pytest.fixture.
# In the notebook we define it as a plain function and call it manually.

@pytest.fixture
def standard_pulse():
    """A canonical 100 fs, 800 nm, 50 µJ pulse for reuse across tests."""
    return GaussianPulse(tau=100e-15, wavelength=800e-9, energy=50e-6)


@pytest.fixture
def time_axis():
    """Standard time grid for FFT-based tests."""
    N, dt = 4096, 5e-15
    return (np.arange(N) - N // 2) * dt


# Use in tests — pytest injects the fixture by name
def test_pulse_intensity_at_zero(standard_pulse):
    """Peak intensity of the normalised envelope is 1."""
    t = np.array([0.0])
    assert standard_pulse.intensity(t)[0] == pytest.approx(1.0)


# Simulate pytest injection in notebook
test_pulse_intensity_at_zero(standard_pulse())
print("Fixture test passed ✓")

In [ ]:
# --------------------------------------------------------------------------
# @pytest.mark.parametrize — run one test over many inputs
# --------------------------------------------------------------------------

TBP_LIMIT = 2 * np.log(2) / np.pi   # 0.4413 for Gaussian pulses


def compute_tbp(tau: float, N: int = 4096, dt: float = 5e-15) -> float:
    """Compute the time-bandwidth product for a Gaussian pulse."""
    t    = (np.arange(N) - N // 2) * dt
    p    = GaussianPulse(tau=tau, wavelength=800e-9, energy=50e-6)
    I_t  = p.intensity(t)
    freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
    S    = np.abs(np.fft.fftshift(np.fft.fft(np.sqrt(I_t))))**2

    def fwhm(x, y):
        yn = y / y.max(); above = yn >= 0.5
        return x[above].max() - x[above].min()

    return fwhm(t, I_t) * fwhm(freq, S)


@pytest.mark.parametrize('tau_fs', [50, 80, 100, 150, 200])
def test_tbp_transform_limited(tau_fs):
    """TBP ≈ 0.4413 regardless of pulse duration."""
    tbp = compute_tbp(tau_fs * 1e-15)
    assert tbp == pytest.approx(TBP_LIMIT, rel=0.01), \
        f"TBP={tbp:.4f} for tau={tau_fs} fs, expected {TBP_LIMIT:.4f}"


# Run manually in notebook
for tau_fs in [50, 80, 100, 150, 200]:
    tbp = compute_tbp(tau_fs * 1e-15)
    print(f"tau={tau_fs} fs: TBP={tbp:.4f} (expected {TBP_LIMIT:.4f}) ✓")

In [ ]:
# --------------------------------------------------------------------------
# conftest.py — project-wide fixtures
# --------------------------------------------------------------------------

# In your repo, create tests/conftest.py:
# ---
# import pytest
# import numpy as np
# from shared.pulse_classes import GaussianPulse
#
# @pytest.fixture
# def standard_pulse():
#     return GaussianPulse(tau=100e-15, wavelength=800e-9, energy=50e-6)
#
# @pytest.fixture
# def time_axis():
#     N, dt = 4096, 5e-15
#     return (np.arange(N) - N // 2) * dt
# ---
# All test files in tests/ automatically have access to these fixtures.

print("conftest.py pattern — see comments above")
print("Recommended repo structure:")
print("""
course/
├── shared/
│   ├── optics.py
│   ├── pulses.py
│   ├── pulse_classes.py
│   └── beam_classes.py
└── tests/
    ├── conftest.py
    ├── test_optics.py
    ├── test_pulses.py
    └── test_pulse_classes.py
""")

---
## Block 6 — Testing classes

In [ ]:
# --------------------------------------------------------------------------
# Testing GaussianPulse — properties, methods, validation
# --------------------------------------------------------------------------

class TestGaussianPulse:
    """Test suite for GaussianPulse."""

    def test_repr(self):
        p = GaussianPulse(100e-15, 800e-9, 50e-6)
        assert '100' in repr(p)   # tau in fs
        assert '800' in repr(p)   # wavelength in nm

    def test_intensity_peak(self):
        p = GaussianPulse(100e-15, 800e-9, 50e-6)
        assert p.intensity(np.array([0.0]))[0] == pytest.approx(1.0)

    def test_intensity_at_tau(self):
        tau = 100e-15
        p   = GaussianPulse(tau, 800e-9, 50e-6)
        I_tau = p.intensity(np.array([tau]))[0]
        assert I_tau == pytest.approx(np.exp(-1), rel=1e-6)

    def test_invalid_tau(self):
        with pytest.raises(PulseError):
            GaussianPulse(tau=-1e-15, wavelength=800e-9, energy=0.0)

    def test_invalid_wavelength(self):
        with pytest.raises(PulseError):
            GaussianPulse(tau=100e-15, wavelength=0.0, energy=0.0)

    def test_zero_energy_allowed(self):
        p = GaussianPulse(100e-15, 800e-9, energy=0.0)
        assert p.energy == 0.0

    def test_equality(self):
        p1 = GaussianPulse(100e-15, 800e-9, 50e-6)
        p2 = GaussianPulse(100e-15, 800e-9, 50e-6)
        # Two independently created pulses with same params
        assert p1.tau == p2.tau
        assert p1.wavelength == p2.wavelength


# Run in notebook
suite = TestGaussianPulse()
for method_name in dir(suite):
    if method_name.startswith('test_'):
        getattr(suite, method_name)()
        print(f"  {method_name} ✓")
print("TestGaussianPulse: all passed ✓")

In [ ]:
# --------------------------------------------------------------------------
# Integration test — OpticalElement composition
# --------------------------------------------------------------------------

class OpticalElement:
    def matrix(self): raise NotImplementedError
    def propagate(self, ray): return self.matrix() @ ray
    def __matmul__(self, other):
        if isinstance(other, OpticalElement):
            return _Composed(self.matrix() @ other.matrix())
        return NotImplemented

class _Composed(OpticalElement):
    def __init__(self, M): self._M = M
    def matrix(self): return self._M

class FreeSpace(OpticalElement):
    def __init__(self, d): self.d = d
    def matrix(self): return np.array([[1.0, self.d],[0.0,1.0]])

class ThinLens(OpticalElement):
    def __init__(self, f): self.f = f
    def matrix(self): return np.array([[1.0,0.0],[-1/self.f,1.0]])


def test_free_space_identity():
    """Propagation over d=0 is the identity matrix."""
    M = FreeSpace(0.0).matrix()
    assert np.allclose(M, np.eye(2))


def test_thin_lens_collimates():
    """A point source at the front focal plane exits as a collimated beam."""
    f   = 0.1
    sys = FreeSpace(f) @ ThinLens(f) @ FreeSpace(f)
    # Diverging ray at y=0: [0, θ]
    ray_in  = np.array([0.0, np.radians(5)])
    ray_out = sys.propagate(ray_in)
    assert ray_out[1] == pytest.approx(0.0, abs=1e-10)   # collimated: θ_out = 0


def test_matrix_composition_associativity():
    """(A @ B) @ C == A @ (B @ C)."""
    A = FreeSpace(0.1)
    B = ThinLens(0.2)
    C = FreeSpace(0.3)
    M1 = ((A @ B) @ C).matrix()
    M2 = (A @ (B @ C)).matrix()
    assert np.allclose(M1, M2)


test_free_space_identity()
test_thin_lens_collimates()
test_matrix_composition_associativity()
print("OpticalElement integration tests passed ✓")

---
## Block 7 — Defensive programming

In [ ]:
# --------------------------------------------------------------------------
# Guard clauses — validate early, return/raise early
# --------------------------------------------------------------------------

# Without guard clauses: deeply nested, hard to read
def compute_tbp_naive(tau, N, dt):
    if tau > 0:
        if N > 0:
            if dt > 0:
                t = (np.arange(N) - N//2) * dt
                # ... actual computation
                return 0.4413
            else:
                return None
        else:
            return None
    else:
        return None


# With guard clauses: flat, readable
def compute_tbp_guarded(tau: float, N: int, dt: float) -> float:
    """Compute TBP for a Gaussian pulse."""
    if tau <= 0:
        raise PulseError('tau', tau, 'must be positive')
    if N <= 0 or not isinstance(N, int):
        raise ValueError(f"N must be a positive integer, got {N}")
    if dt <= 0:
        raise ValueError(f"dt must be positive, got {dt}")

    t    = (np.arange(N) - N//2) * dt
    I    = np.exp(-t**2 / tau**2)
    freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
    S    = np.abs(np.fft.fftshift(np.fft.fft(np.sqrt(I))))**2

    def fwhm(x, y):
        yn = y / y.max(); above = yn >= 0.5
        return x[above].max() - x[above].min()

    return fwhm(t, I) * fwhm(freq, S)


print(f"TBP = {compute_tbp_guarded(100e-15, 4096, 5e-15):.4f}")

In [ ]:
# --------------------------------------------------------------------------
# warnings.warn — soft validation, deprecation
# --------------------------------------------------------------------------

import warnings

def gaussian_intensity_v2(
    r: np.ndarray,
    w: float,
    P: float,
    power: float = None,   # deprecated parameter name
) -> np.ndarray:
    """
    Radial Gaussian beam intensity.

    .. deprecated::
       The `power` parameter is deprecated; use `P` instead.
    """
    if power is not None:
        warnings.warn(
            "The 'power' parameter is deprecated; use 'P' instead.",
            DeprecationWarning,
            stacklevel=2,
        )
        P = power

    if w <= 0:
        warnings.warn(
            f"Beam radius w={w:.2e} is non-positive; returning zeros.",
            UserWarning,
            stacklevel=2,
        )
        return np.zeros_like(r)

    I0 = 2 * P / (np.pi * w**2)
    return I0 * np.exp(-2 * r**2 / w**2)


# Trigger the deprecation warning
r = np.array([0.0, 500e-6])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    I = gaussian_intensity_v2(r, w=500e-6, P=1.0, power=1.0)

print(f"Warnings caught: {len(caught)}")
if caught:
    print(f"  {caught[0].category.__name__}: {caught[0].message}")

In [ ]:
# --------------------------------------------------------------------------
# assert in production code — internal invariants only
# --------------------------------------------------------------------------

def propagate_pulse(E_f: np.ndarray, phase: np.ndarray) -> np.ndarray:
    """
    Apply a spectral phase to a pulse field.

    Uses assert for internal consistency checks (developer errors),
    not for user input validation.
    """
    # Internal invariant — asserts are for developer errors, not user input
    assert E_f.shape == phase.shape, (
        f"Shape mismatch: E_f={E_f.shape}, phase={phase.shape}"
    )
    assert np.isrealobj(phase), "Phase must be real-valued"

    return E_f * np.exp(1j * phase)


# ⚠️ assert is disabled with python -O
# Never use assert for user-facing input validation or security checks

N    = 256
E_f  = np.ones(N, dtype=complex)
phi  = np.linspace(0, np.pi, N)
E_out = propagate_pulse(E_f, phi)
print(f"propagate_pulse: output shape {E_out.shape}, |E|={np.abs(E_out).mean():.4f}")

In [ ]:
# --------------------------------------------------------------------------
# Logging — structured output for long-running computations
# --------------------------------------------------------------------------

import logging

# Configure once at module level (or in main script)
logging.basicConfig(
    level   = logging.DEBUG,
    format  = '%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt = '%H:%M:%S',
)
logger = logging.getLogger('shared.optics')


def propagate_beam(
    beam_radius: float,
    elements: list,
) -> float:
    """Propagate a beam through a sequence of optical elements."""
    logger.debug("Starting propagation: w_in=%.2f µm, %d elements",
                 beam_radius * 1e6, len(elements))

    ray = np.array([beam_radius, 0.0])
    for i, el in enumerate(elements):
        ray = el.propagate(ray)
        logger.debug("After element %d: y=%.3f mm, θ=%.4f mrad",
                     i, ray[0]*1e3, ray[1]*1e3)

        if abs(ray[0]) > 0.1:   # beam clipped
            logger.warning("Beam radius %.1f mm may exceed aperture", ray[0]*1e3)

    logger.info("Propagation complete: w_out=%.2f µm", abs(ray[0])*1e6)
    return abs(ray[0])


elements = [FreeSpace(0.1), ThinLens(0.1), FreeSpace(0.2)]
w_out    = propagate_beam(2e-3, elements)
print(f"\nOutput beam radius: {w_out*1e3:.2f} mm")

💡 **Pro Tip — logging levels**

| Level | Use for |
|-------|---------|
| `DEBUG` | Detailed internal state, per-step values |
| `INFO` | Milestone messages (start, end, key results) |
| `WARNING` | Something unexpected but recoverable |
| `ERROR` | A failure that should not have happened |
| `CRITICAL` | System-level failure |

In production: set `level=logging.WARNING` to suppress debug noise.  
In development: set `level=logging.DEBUG` to see everything.  
Never use `print()` in library code — use `logging`.

---
## Block 8 — Coverage and CI preview

In [ ]:
# --------------------------------------------------------------------------
# Running coverage from the terminal
# --------------------------------------------------------------------------

# Install:
#   pip install pytest-cov
#
# Run with coverage:
#   pytest tests/ --cov=shared --cov-report=term-missing
#
# Output:
#   Name                    Stmts   Miss  Cover   Missing
#   -------------------------------------------------------
#   shared/optics.py           24      3    88%    45-47
#   shared/pulses.py           18      0   100%
#   shared/pulse_classes.py    52      6    88%    91-96
#   -------------------------------------------------------
#   TOTAL                      94      9    90%
#
# HTML report (open in browser):
#   pytest tests/ --cov=shared --cov-report=html
#   open htmlcov/index.html

print("Coverage commands — run these from the terminal in your repo root:")
print()
print("  pytest tests/ -v")
print("  pytest tests/ --cov=shared --cov-report=term-missing")
print("  pytest tests/ --cov=shared --cov-report=html")

In [ ]:
# --------------------------------------------------------------------------
# GitHub Actions CI — preview (full implementation in S10)
# --------------------------------------------------------------------------

# .github/workflows/tests.yml
CI_YAML = """
name: Tests

on: [push, pull_request]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install numpy scipy matplotlib pytest pytest-cov
      - run: pytest tests/ --cov=shared --cov-report=term-missing
"""

print("GitHub Actions workflow (preview — implemented in S10):")
print(CI_YAML)

---
## Summary

| Concept | Key syntax |
|---------|------------|
| Type hint | `def f(x: float) -> np.ndarray:` |
| Optional | `Optional[float]` or `float \| None` (Python 3.10+) |
| Forward ref | `from __future__ import annotations` |
| Protocol | `class P(Protocol): def method(self, ...): ...` |
| Raise | `raise ValueError("msg")` |
| Chain | `raise NewError(...) from original_exc` |
| Catch | `try: ... except ValueError as e: ...` |
| Custom exc | `class PulseError(PhysicsError, ValueError): ...` |
| Test function | `def test_foo(): assert ...` |
| Float compare | `assert x == pytest.approx(y, rel=1e-4)` |
| Exc test | `with pytest.raises(ValueError, match='pattern'):` |
| Fixture | `@pytest.fixture` — injected by name |
| Parametrize | `@pytest.mark.parametrize('x', [1, 2, 3])` |
| Soft warn | `warnings.warn('msg', DeprecationWarning, stacklevel=2)` |
| Logging | `logger = logging.getLogger(__name__)` |
| Coverage | `pytest --cov=shared --cov-report=term-missing` |

**No new `shared/` files** — this session hardens existing code.  
Deliverable: a `tests/` directory in the repo with working `pytest` suites.

**Next: S09 — Performance & FFI**